In [11]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from scipy.stats import fisher_exact
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import proportion_confint
from patsy.contrasts import Treatment as PatsyTreatment
from pathlib import Path
from collections import defaultdict

In [ ]:
# -------------------------------------------------
# 1 · Compute counts ── raw tallies for every product
# -------------------------------------------------

# ---------- CONFIG ----------
CSV_DIR = Path("csv-files")          # adapt if your CSVs live elsewhere
FILE_MAP = {
    "WebMD":  CSV_DIR / "Kidney Stone Reviews - Reviews - WebMD.csv",
    "Amazon": CSV_DIR / "Kidney Stone Reviews - Reviews - Amazon.csv",
    "Reddit": CSV_DIR / "Kidney Stone Reviews - Reviews - Reddit.csv",
}
HELP_COL = "Helps overall with kidney stones"
AE_COL   = "Side effects mentioned"
# -----------------------------

def _clean_help_col(series: pd.Series) -> pd.Series:
    """
    Normalise HELP_COL so we can separate ‘no information’ from explicit ‘no’.
    Assumes:  1 = helped, 0 or NaN = no info, −1 = explicitly said it did NOT help.
    If your coding scheme is different, adjust the mapping below.
    """
    mapping = {True: 1, False: -1}          # in case booleans slipped in
    return (series.replace(mapping)
                  .where(series.isin([1, -1]))    # leave 1 / -1 unchanged
                  .astype("Int64"))                # keep NA as <NA>

def load_kidney_stone_data() -> dict[str, pd.DataFrame]:
    """Read the three CSVs into a dict of DataFrames and make sure key columns are typed."""
    data = {}
    for platform, path in FILE_MAP.items():
        df = pd.read_csv(path)
        # Normalise the two key columns
        df[HELP_COL] = _clean_help_col(df[HELP_COL])
        # AE column is 1 if any text is present, else 0
        df[AE_COL] = df[AE_COL].apply(lambda x: 1 if isinstance(x, str) and x.strip() else 0)
        data[platform] = df
    return data

def compute_platform_counts(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return counts by Medicine for a single platform.
    Columns: N, Helped, Not helped, No info, Adverse events.
    """
    g = df.groupby("Medicine")
    counts = g.size().to_frame(name="N")
    helped      = g[HELP_COL].apply(lambda s: (s == 1).sum()).rename("Helped")
    not_helped  = g[HELP_COL].apply(lambda s: (s == -1).sum()).rename("Not helped")
    adverse_evt = g[AE_COL].sum().rename("Adverse events")
    counts = counts.join([helped, not_helped, adverse_evt])
    counts["No info"] = counts["N"] - counts["Helped"] - counts["Not helped"]
    return counts[["N", "Helped", "Not helped", "No info", "Adverse events"]].sort_index()

def compute_all_counts() -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    """
    Returns:
      • dict  counts_by_platform[platform] -> DataFrame
      • DataFrame grand_total  (sums across platforms)
    """
    data = load_kidney_stone_data()
    counts_by_platform = {p: compute_platform_counts(df) for p, df in data.items()}
    # grand‑total = sum DataFrames, re‑index so missing products get filled with 0
    grand_total = pd.concat(counts_by_platform.values(), axis=0) \
                     .groupby(level=0) \
                     .sum() \
                     .astype(int)
    return counts_by_platform, grand_total

platform_tables, overall_table = compute_all_counts()

# Show the WebMD table
display(platform_tables["WebMD"])

# Save CSVs for later steps (forest plot, modelling, etc.)
for plat, tbl in platform_tables.items():
    tbl.to_csv(CSV_DIR / f"{plat.lower()}_raw_counts.csv", index=True)
overall_table.to_csv(CSV_DIR / "all_platforms_raw_counts.csv", index=True)
print("✔  Raw‑count tables saved.")

In [9]:
# -------------------------------------------------
# 2 · Add Wilson 95 % CIs  ➜  effect‑estimate tables
# -------------------------------------------------

def add_wilson(df, num_col, denom_col, prefix):
    """Append % and Wilson CI columns to df and return it."""
    low, high = proportion_confint(df[num_col], df[denom_col], method="wilson")
    df[f"{prefix} %"]       = 100 * df[num_col] / df[denom_col]
    df[f"{prefix} CI low"]  = 100 * low
    df[f"{prefix} CI high"] = 100 * high
    return df

def build_effect_table(raw_counts_path):
    tbl = pd.read_csv(raw_counts_path, index_col=0)
    tbl = add_wilson(tbl, "Helped",         "N", "Helped")
    tbl = add_wilson(tbl, "Adverse events", "N", "AE")
    cols = ["N",
            "Helped %", "Helped CI low", "Helped CI high",
            "AE %",     "AE CI low",     "AE CI high"]
    return tbl[cols].round(1).sort_index()

# ---------- run for each platform ----------
for csv_path in CSV_DIR.glob("*_raw_counts.csv"):
    effect_tbl = build_effect_table(csv_path)
    out_path = csv_path.with_name(csv_path.stem.replace("_raw_counts",
                                                       "_effect_estimates") + ".csv")
    effect_tbl.to_csv(out_path)
    print("saved ➜", out_path.name)

# ---------- aggregate (all platforms) ----------
agg_raw = CSV_DIR / "all_platforms_raw_counts.csv"
agg_out = CSV_DIR / "all_platforms_effect_estimates.csv"
build_effect_table(agg_raw).to_csv(agg_out)
print("saved ➜", agg_out.name)

# Peek at the aggregate result
pd.read_csv(agg_out, index_col=0).head()


saved ➜ all_platforms_effect_estimates.csv
saved ➜ reddit_effect_estimates.csv
saved ➜ webmd_effect_estimates.csv
saved ➜ amazon_effect_estimates.csv
saved ➜ all_platforms_effect_estimates.csv


,N,Helped %,Helped CI low,Helped CI high,AE %,AE CI low,AE CI high
Medicine,,,,,,,
Allopurinol,62,24.2,15.2,36.2,21.0,12.7,32.6
Ashwagandha,271,0.0,0.0,1.4,48.0,42.1,53.9
Black seed,94,4.3,1.7,10.4,21.3,14.2,30.6
Chanca piedra,1772,53.3,50.9,55.6,5.7,4.7,6.9
Flomax,1156,24.3,21.9,26.9,23.2,20.8,25.7


In [17]:
# -------------------------------------------------
# 3 · Forest plots (effectiveness & adverse events)
# -------------------------------------------------
# Uses Plotly so the same figure works interactively in Jupyter *and*
# can be exported to PNG/SVG if you have `kaleido` installed.
# -------------------------------------------------

def forest_plot(table_path: Path,
                prop_col: str, low_col: str, high_col: str,
                title: str, file_stub: str,
                colour: str = "black"):
    """
    Horizontal forest plot for a single CSV.
      • Drops rows with prop == 0
      • X‑axis locked to [0, 100]
      • colour = marker colour (useful for multi‑platform plot)
    """
    import plotly.graph_objects as go

    df = (pd.read_csv(table_path, index_col=0)
            .query(f"`{prop_col}` > 0")       # exclude zeros
            .sort_values(prop_col))

    err_minus = (df[prop_col] - df[low_col]).clip(lower=0)
    err_plus  = (df[high_col] - df[prop_col]).clip(lower=0)

    fig = go.Figure(go.Scatter(
        x=df[prop_col],
        y=df.index,
        mode="markers",
        marker=dict(symbol="square", size=10, color=colour),
        error_x=dict(type="data",
                     symmetric=False,
                     array=err_plus,
                     arrayminus=err_minus,
                     thickness=1, width=0)))
    fig.update_layout(
        title=title,
        xaxis=dict(title="Proportion (%)", range=[0, 100]),
        yaxis=dict(autorange="reversed"),
        template="simple_white",
        margin=dict(l=140, r=20, t=80, b=40)
    )

    html_path = CSV_DIR / f"{file_stub}.html"
    fig.write_html(html_path)

    try:
        png_path = CSV_DIR / f"{file_stub}.png"
        fig.write_image(png_path, scale=2)
        print(f"✓ saved {html_path.name} and {png_path.name}")
    except ValueError:
        print(f"✓ saved {html_path.name}  (install 'kaleido' for PNG export)")

    fig.show()

# -------------------------------------------------
# Forest plots for each platform separately
# -------------------------------------------------
for csv_path in CSV_DIR.glob("*_effect_estimates.csv"):
    if csv_path.stem.startswith("all_platforms"):
        continue
    plat = csv_path.stem.replace("_effect_estimates", "").capitalize()
    colour = dict(Webmd="#636EFA", Amazon="#EF553B", Reddit="#00CC96").get(plat, "black")

    forest_plot(csv_path,
                prop_col="Helped %",
                low_col="Helped CI low",
                high_col="Helped CI high",
                title=f"{plat}: proportion reporting help",
                file_stub=f"forest_{plat.lower()}_helped",
                colour=colour)

    forest_plot(csv_path,
                prop_col="AE %",
                low_col="AE CI low",
                high_col="AE CI high",
                title=f"{plat}: proportion reporting adverse events",
                file_stub=f"forest_{plat.lower()}_ae",
                colour=colour)

# -------------------------------------------------
# Combined forest plot: WebMD, Amazon, Reddit on one figure
# -------------------------------------------------
def combined_forest(prop_col, low_col, high_col,
                    title, file_stub):

    import plotly.graph_objects as go
    traces = []
    colour_map = {"Webmd": "#636EFA",
                  "Amazon": "#EF553B",
                  "Reddit": "#00CC96"}

    # ---------- build a tidy dataframe ----------
    records = []
    for csv_path in CSV_DIR.glob("*_effect_estimates.csv"):
        if csv_path.stem.startswith("all_platforms"):
            continue 
        plat = csv_path.stem.replace("_effect_estimates", "").capitalize()

        # 1. read → 2. copy index to column ‘Product’ → 3. drop zeros
        df = pd.read_csv(csv_path, index_col=0)
        df["Product"] = df.index           # <‑‑ key line adds the column
        df["Platform"] = plat
        df = df[df[prop_col] > 0]          # drop products that are 0 %

        records.append(df[["Product", "Platform", prop_col,
                           low_col, high_col]])

    tidy = pd.concat(records, ignore_index=True)

    # choose product order by *average* proportion across platforms
    order = (tidy.groupby("Product")[prop_col]
                  .mean()
                  .sort_values()
                  .index.tolist())
    tidy["ypos"] = tidy["Product"].apply(order.index)

    fig = go.Figure()
    for plat, sub in tidy.groupby("Platform"):
        err_minus = (sub[prop_col] - sub[low_col]).clip(lower=0)
        err_plus  = (sub[high_col] - sub[prop_col]).clip(lower=0)

        fig.add_trace(go.Scatter(
            x=sub[prop_col],
            y=sub["ypos"],
            mode="markers",
            name=plat,
            marker=dict(symbol="square", size=10,
                        color=colour_map.get(plat, "black")),
            error_x=dict(type="data",
                         symmetric=False,
                         array=err_plus,
                         arrayminus=err_minus,
                         thickness=1, width=0)))

    fig.update_layout(
        title=title,
        xaxis=dict(title="Proportion (%)", range=[0, 100]),
        yaxis=dict(tickmode="array",
                   tickvals=list(range(len(order))),
                   ticktext=order,
                   autorange="reversed"),
        template="simple_white",
        margin=dict(l=160, r=20, t=80, b=40)
    )

    html_path = CSV_DIR / f"{file_stub}.html"
    fig.write_html(html_path)
    try:
        png_path = CSV_DIR / f"{file_stub}.png"
        fig.write_image(png_path, scale=2)
        print(f"✓ saved {html_path.name} and {png_path.name}")
    except ValueError:
        print(f"✓ saved {html_path.name}  (install 'kaleido' for PNG export)")

    fig.show()

# build combined plots for both outcomes
combined_forest("Helped %", "Helped CI low", "Helped CI high",
                title="Proportion reporting help (all platforms)",
                file_stub="forest_combined_helped")

combined_forest("AE %", "AE CI low", "AE CI high",
                title="Proportion reporting adverse events (all platforms)",
                file_stub="forest_combined_ae")


✓ saved forest_webmd_helped.html and forest_webmd_helped.png


✓ saved forest_webmd_ae.html and forest_webmd_ae.png


✓ saved forest_amazon_helped.html and forest_amazon_helped.png


✓ saved forest_amazon_ae.html and forest_amazon_ae.png


✓ saved forest_reddit_helped.html and forest_reddit_helped.png


✓ saved forest_reddit_ae.html and forest_reddit_ae.png


✓ saved forest_combined_helped.html and forest_combined_helped.png


✓ saved forest_combined_ae.html and forest_combined_ae.png


In [13]:
# ==================================================
# 4 · Random‑effects meta‑analysis (platform = study)
#     Outputs pooled proportion + 95 % CI + I² (%)
# ==================================================

# ---------- helper functions ----------
def logit(p):
    return np.log(p / (1 - p))

def inv_logit(l):
    return 1 / (1 + np.exp(-l))

def dl_random_effects(y, v):
    """
    DerSimonian‑Laird estimator.
      y : array of study logit proportions
      v : array of within‑study variances
    Returns pooled effect, pooled var, tau², Q, I².
    """
    w = 1 / v
    y_fixed = np.sum(w * y) / np.sum(w)
    Q = np.sum(w * (y - y_fixed) ** 2)
    df = len(y) - 1
    c = np.sum(w) - np.sum(w**2) / np.sum(w)
    tau2 = max(0, (Q - df) / c)         # between‑study variance
    w_re = 1 / (v + tau2)
    y_pool = np.sum(w_re * y) / np.sum(w_re)
    var_pool = 1 / np.sum(w_re)
    I2 = max(0, (Q - df) / Q) * 100 if Q > df else 0
    return y_pool, var_pool, tau2, Q, I2

# ---------- load raw counts we saved in Step 1 ----------
plat_raw = {
    p.stem.replace("_raw_counts", "").capitalize(): pd.read_csv(p, index_col=0)
    for p in CSV_DIR.glob("*_raw_counts.csv")
    if not p.stem.startswith("all_platforms")
}

products = sorted(
    {prod for tbl in plat_raw.values() for prod in tbl.index}
    - {"Ashwagandha", "Melatonin"}        # optional exclusion
)

def meta_analysis(outcome_col, csv_name):
    rows = []
    for drug in products:
        yi, vi, ns = [], [], []           # study effect, variance, sample size
        avail_platforms = []
        for plat, tbl in plat_raw.items():
            if drug not in tbl.index:            # not reviewed on that platform
                continue
            n  = tbl.loc[drug, "N"]
            x  = tbl.loc[drug, outcome_col]      # Helped or Adverse events
            # continuity correction for 0 or n
            x_adj = x + 0.5 if x in (0, n) else x
            n_adj = n + 1             if x in (0, n) else n
            p   = x_adj / n_adj
            yi.append(logit(p))
            vi.append(1 / (x_adj) + 1 / (n_adj - x_adj))  # variance of logit(p)
            ns.append(n)
            avail_platforms.append(plat)

        if len(yi) < 2:                          # need ≥2 studies for random‑effects
            continue
        y_pool, var_pool, tau2, Q, I2 = dl_random_effects(np.array(yi), np.array(vi))
        pooled_p = inv_logit(y_pool)
        se_pool  = np.sqrt(var_pool)
        ci_low, ci_high = inv_logit(y_pool - 1.96 * se_pool), inv_logit(y_pool + 1.96 * se_pool)

        rows.append({
            "Product": drug,
            "K platforms": len(yi),
            "Total N": sum(ns),
            "Pooled %": round(100 * pooled_p, 1),
            "95% CI low": round(100 * ci_low, 1),
            "95% CI high": round(100 * ci_high, 1),
            "Tau²": round(tau2, 4),
            "I² (%)": round(I2, 1),
            "Q": round(Q, 2)
        })

    out_df = pd.DataFrame(rows).set_index("Product").sort_values("Pooled %", ascending=False)
    out_path = CSV_DIR / csv_name
    out_df.to_csv(out_path)
    print("✓ saved", csv_name)
    return out_df

meta_help = meta_analysis("Helped",         "meta_helped_random_effects.csv")
meta_ae   = meta_analysis("Adverse events", "meta_adverse_events_random_effects.csv")

display(meta_help)


✓ saved meta_helped_random_effects.csv
✓ saved meta_adverse_events_random_effects.csv


,K platforms,Total N,Pooled %,95% CI low,95% CI high,Tau²,I² (%),Q
Product,,,,,,,,
Phosfood,2,42,75.5,60.6,86.1,0.0000,0.0,0.10
Rowatinex,2,111,64.6,5.6,98.2,5.8917,96.7,30.48
Chanca piedra,3,1772,61.3,39.6,79.3,0.5737,98.1,104.04
Potassium citrate,3,666,55.7,10.7,92.9,4.2161,98.4,125.68
Hydrochlorothiazide,2,66,33.3,23.1,45.5,0.0000,0.0,0.04
Allopurinol,2,62,26.4,13.3,45.5,0.1716,43.2,1.76
Flomax,2,1156,18.5,7.3,39.8,0.4220,60.3,2.52
Black seed,2,94,5.3,0.1,70.5,6.2983,84.3,6.38
Garcinia,2,930,2.3,0.0,96.3,24.6407,95.9,24.37
